# 01 — Feature engineering du churn

**Uniquement la préparation des features :** lecture et qualité, transformations, sélection des colonnes, partitions et sauvegarde. Aucun modèle prédictif, entraînement CatBoost, seuil de classification ou suivi d'entraînement MLflow ici.

`ChurnFeatureEngineer` reste le point d'entrée unique pour toutes les transformations. Ses bornes tarifaires sont ajustées sur le train uniquement. Les données préparées passent au **02 — Modélisation**, puis le modèle sauvegardé passe au **03 — Évaluation et explicabilité**.

Exécuter ce notebook avant le 02, dans un noyau Python 3.12. Les trois CSV principaux restent dans `data/processed/`. La description des features, les rapports et les entrées préparées des diagnostics restent dans `data/interim/features/`. La préparation des partitions de diagnostic ne lance aucun modèle.

## 1 — Imports et paramètres

La sélection de groupes et de périodes prépare uniquement des partitions de données.

In [15]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.model_selection import GroupShuffleSplit

if not Path("settings.py").is_file() and Path("../settings.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from settings import settings

SEED = 42
TRAIN_MONTHS = ("2024-01", "2024-07")
VALIDATION_MONTHS = ("2024-08", "2024-09")
TEST_MONTHS = ("2024-10", "2024-11")
WALK_FORWARD_MONTHS = ("2024-08", "2024-11")
PREPARATION_DIR = settings.PROJECT_ROOT / "data" / "interim" / "features"
FEATURE_NOTEBOOK_PATH = settings.PROJECT_ROOT / "notebooks" / "01_churn_feature_engineering.ipynb"
np.random.seed(SEED)
print("Source :", settings.DATA_PATH)

Source : /Users/surelmanda/Downloads/sujet-entretien-technique/Telecom-Retention-Optimize/data/raw/churn_clients.csv


## 2 — Lecture et contrôle de qualité

Une ligne = un client observé en fin de mois. `churn=1` indique une résiliation **dans les 30 jours suivants**. Le même client peut apparaître sur plusieurs mois ; un client résilié disparaît ensuite. Nous n’utilisons ni `customer_id` comme prédicteur ni les labels des mois futurs.

### Traitement explicite des doublons

Une ligne doit correspondre à **un client et une date de snapshot uniques**. On distingue :

- **Copies exactes** : suppression des copies redondantes, sans perte d'information.
- **Même client × snapshot, valeurs différentes** : aucune ligne n'est choisie arbitrairement. **Toutes les lignes de la clé ambiguë sont exclues de la modélisation**, avec un diagnostic affiché. Cela peut réduire l'échantillon ou introduire un biais de sélection : le nombre de cas concernés doit être examiné avant de valider le modèle.

Cette politique conservatrice est adaptée à une première exécution ; si les conflits sont nombreux, il faut revenir à la source pour déterminer la règle métier correcte (version des données, horodatage de mise à jour, etc.). **Ne jamais dédupliquer sur `customer_id` seul** : les snapshots mensuels successifs sont attendus.

In [16]:
df = pd.read_csv(settings.DATA_PATH, dtype={"customer_id": "string", "dept": "string"})

required = [
    "customer_id",
    "snapshot_date",
    "signup_date",
    "age",
    "contract",
    "region",
    "dept",
    "channel",
    "monthly_fee",
    "data_gb",
    "support_calls_3m",
    "late_payments",
    "nps",
    "last_invoice_amount",
    "retention_offer_declined",
    "churn",
]
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f"Colonnes absentes : {missing_columns}")

df["customer_id"] = df["customer_id"].str.strip()

for column in ["snapshot_date", "signup_date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

numeric_columns = [
    "age",
    "monthly_fee",
    "data_gb",
    "support_calls_3m",
    "late_payments",
    "nps",
    "last_invoice_amount",
    "churn",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in ["contract", "region", "dept", "channel"]:
    df[column] = df[column].astype("string").str.strip().str.strip("\"'").str.lower()

if df["snapshot_date"].isna().any():
    raise ValueError("Des snapshot_date sont invalides.")
if df["customer_id"].isna().any() or df["customer_id"].eq("").any():
    raise ValueError("Des identifiants client sont manquants.")
# Audit et résolution prudente des doublons client × mois.
# Les copies parfaitement identiques sont dédupliquées.
# Les clés présentant des valeurs divergentes sont exclues (jamais arbitrées au hasard).
KEY = ["customer_id", "snapshot_date"]

raw_count = len(df)
df = df.drop_duplicates().copy()
exact_removed = raw_count - len(df)

ambiguous_mask = df.duplicated(KEY, keep=False)
ambiguous_rows = df.loc[ambiguous_mask].copy()
ambiguous_keys = ambiguous_rows[KEY].drop_duplicates()

print(f"Lignes chargées : {raw_count:,}")
print(f"Copies strictement identiques supprimées : {exact_removed:,}")
print(f"Clés client × snapshot ambiguës : {len(ambiguous_keys):,}")

if not ambiguous_rows.empty:
    # Diagnostic de conflits de label, sans modifier la cible.
    label_conflicts = (
        ambiguous_rows.groupby(KEY, dropna=False)["churn"].nunique(dropna=False).gt(1).sum()
    )
    print(f"Clés avec labels contradictoires ou manquants : {label_conflicts:,}")
    print(f"Lignes ambiguës exclues : {len(ambiguous_rows):,}")
    display(ambiguous_rows.sort_values(KEY).head(12))

    # Une clé ambiguë entière est exclue pour éviter une sélection arbitraire.
    df = df.loc[~ambiguous_mask].copy()

if df.empty:
    raise ValueError("Aucune observation non ambiguë après audit des doublons.")

assert not df.duplicated(KEY).any()
print(f"Lignes conservées après audit : {len(df):,}")
if not df["churn"].dropna().isin([0, 1]).all():
    raise ValueError("churn doit être binaire (0 ou 1).")

labeled = df.dropna(subset=["churn"]).copy()
if labeled.empty:
    raise ValueError("Aucune observation avec un label churn valide.")
labeled["churn"] = labeled["churn"].astype(int)

print("Snapshots étiquetés :", len(labeled))
print("Clients distincts :", labeled["customer_id"].nunique())
print("Prévalence churn :", f"{labeled['churn'].mean():.2%}")
print(
    "Période :", labeled["snapshot_date"].min().date(), "→", labeled["snapshot_date"].max().date()
)

display(labeled.head(3))

Lignes chargées : 38,689
Copies strictement identiques supprimées : 60
Clés client × snapshot ambiguës : 0
Lignes conservées après audit : 38,629
Snapshots étiquetés : 38629
Clients distincts : 4000
Prévalence churn : 3.01%
Période : 2024-01-31 → 2024-12-31


,customer_id,snapshot_date,signup_date,age,contract,region,dept,channel,monthly_fee,data_gb,support_calls_3m,late_payments,nps,last_invoice_amount,retention_offer_declined,churn
0,102683,2024-10-31,2020-12-04,38.0,biannuel,occ,02,partenaire,65.15,7.9,1,1,7.0,65.15,0,0
1,100127,2024-04-30,2020-05-25,40.0,mensuel,ara,01,telephone,48.21,14.2,0,0,NaN,48.21,0,0
2,102042,2024-04-30,2024-01-06,21.0,mensuel,idf,09,partenaire,49.09,5.8,1,0,10.0,49.09,0,0


### 2.1 — Vérifier le panel et les churns

L'identifiant client n'est pas censé être unique : l'unité statistique est **(client, snapshot)**. En revanche, le couple doit être unique. Nous contrôlons les observations post-churn, les churns multiples et la différence entre **29,05 % de clients ayant résilié sur la période** et la **prévalence par snapshot** (environ 3,01 % selon les chiffres communiqués). Les valeurs exactes sont recalculées.

In [17]:
customer_counts = labeled.groupby("customer_id").size()
key_duplicates = labeled.duplicated(["customer_id", "snapshot_date"], keep=False).sum()
panel_sorted = labeled.sort_values(["customer_id", "snapshot_date"]).copy()
panel_sorted["prior_churn"] = panel_sorted.groupby("customer_id")["churn"].transform(
    lambda s: s.shift(fill_value=0).cummax()
)
post_churn = panel_sorted.loc[panel_sorted["prior_churn"].eq(1)]
churn_per_customer = labeled.groupby("customer_id")["churn"].sum()
panel_audit = pd.DataFrame(
    {
        "Contrôle": [
            "Snapshots",
            "Clients distincts",
            "Clients répétés",
            "Clés client × mois dupliquées",
            "Observations après churn",
            "Clients avec > 1 churn",
            "Clients avec 1 churn",
            "Clients sans churn observé",
            "Prévalence par snapshot",
        ],
        "Valeur": [
            len(labeled),
            labeled.customer_id.nunique(),
            int((customer_counts > 1).sum()),
            int(key_duplicates),
            len(post_churn),
            int((churn_per_customer > 1).sum()),
            int((churn_per_customer == 1).sum()),
            int((churn_per_customer == 0).sum()),
            round(labeled.churn.mean(), 5),
        ],
    }
)
display(panel_audit)
assert key_duplicates == 0, "Clés client × snapshot répétées"
assert post_churn.empty, "Snapshots post-résiliation à investiguer"
assert not (churn_per_customer > 1).any(), "Plusieurs résiliations par client"
fig = px.histogram(
    customer_counts.rename("snapshots").reset_index(),
    x="snapshots",
    nbins=12,
    title="Nombre de snapshots observés par client",
    color_discrete_sequence=["#2563EB"],
)
fig.update_layout(
    template="plotly_white",
    height=370,
    xaxis_title="Snapshots par client",
    yaxis_title="Nombre de clients",
)
fig.show(config={"responsive": True})

,Contrôle,Valeur
0,Snapshots,38629.00000
1,Clients distincts,4000.00000
2,Clients répétés,3844.00000
3,Clés client × mois dupliquées,0.00000
4,Observations après churn,0.00000
5,Clients avec > 1 churn,0.00000
6,Clients avec 1 churn,1162.00000
7,Clients sans churn observé,2838.00000
8,Prévalence par snapshot,0.03008


### 2.2 — Volumes et cible par mois

Un modèle entraîné sur une année unique peut être sensible aux variations mensuelles. Avant le split, vérifier la présence des deux classes et les volumes par mois.

In [18]:
monthly = (
    labeled.assign(month=labeled["snapshot_date"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)
    .agg(n=("churn", "size"), churn_rate=("churn", "mean"), positives=("churn", "sum"))
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=monthly["month"], y=monthly["n"], name="Snapshots", marker_color="#93C5FD"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=monthly["month"],
        y=monthly["churn_rate"],
        name="Taux de churn",
        mode="lines+markers",
        line=dict(color="#E11D48", width=3),
    ),
    secondary_y=True,
)
fig.update_yaxes(title_text="Nombre de snapshots", secondary_y=False)
fig.update_yaxes(title_text="Taux de churn", tickformat=".0%", secondary_y=True)
fig.update_layout(
    title="Volumes et churn observés par mois",
    template="plotly_white",
    height=420,
    hovermode="x unified",
)
fig.show(config={"responsive": True})

## 3 — Catalogue des 22 variables métier

Les variables ci-dessous proviennent exclusivement d’informations disponibles à la date du snapshot **sous réserve de confirmation de leur temporalité**. Les seuils sont des hypothèses métier, pas des résultats optimisés sur le test.

| # | Feature | Logique métier | Formule |
|---:|---|---|---|
| 1 | `tenure_months` | Ancienneté à la date du snapshot | `(snapshot_date − signup_date).days / 30.4375` |
| 2 | `tenure_band` | Phase du cycle de vie | Classe de `tenure_months` : `[0,3[`, `[3,6[`, `[6,12[`, `[12,24[`, `[24,48[`, `[48,+∞[` |
| 3 | `is_new_customer` | Moins de 3 mois d’ancienneté | `1[tenure_months < 3]` |
| 4 | `has_support_calls` | A sollicité le support | `1[support_calls_3m > 0]` |
| 5 | `high_support_calls` | Au moins 3 appels au support | `1[support_calls_3m ≥ 3]` |
| 6 | `has_late_payments` | Au moins un retard de paiement | `1[late_payments > 0]` |
| 7 | `support_payment_interaction` | Appels × retards | `support_calls_3m × late_payments` |
| 8 | `high_friction` | Support élevé ET retard | `1[(support_calls_3m ≥ 3) ET (late_payments ≥ 1)]` |
| 9 | `nps_segment` | Détracteur, passif, promoteur | `detractor` si `0 ≤ nps ≤ 6` ; `passive` si `6 < nps ≤ 8` ; `promoter` si `8 < nps ≤ 10` |
| 10 | `support_dissatisfaction` | Appels × (10 − NPS) | `support_calls_3m × (10 − nps)` |
| 11 | `nps_missing` | NPS absent ou invalide | `1[nps manquant OU nps < 0 OU nps > 10]` |
| 12 | `invoice_fee_gap` | Dernière facture − mensualité | `last_invoice_amount − monthly_fee` |
| 13 | `invoice_fee_ratio` | Dernière facture / mensualité | `last_invoice_amount / monthly_fee` si `monthly_fee > 0`, sinon manquant |
| 14 | `fee_per_gb` | Mensualité / Go consommés | `monthly_fee / data_gb` si `data_gb > 0`, sinon manquant |
| 15 | `low_usage_flag` | Consommation inférieure à 1 Go | `1[data_gb < 1]` |
| 16 | `contract_tenure` | Contrat × cycle de vie | `contract + "__" + tenure_band` |
| 17 | `contract_channel` | Contrat × canal | `contract + "__" + channel` |
| 18 | `contract_region` | Contrat × région | `contract + "__" + region` |
| 19 | `payment_invoice_interaction` | Retards × dépassement de facture | `late_payments × max(last_invoice_amount − monthly_fee, 0)` |
| 20 | `invoice_below_fee` | Facture inférieure à la mensualité | `1[last_invoice_amount < monthly_fee]` |
| 21 | `monthly_fee_band` | Tranche tarifaire relative au train | `pd.cut(monthly_fee, [-∞, q₁/₃(train), q₂/₃(train), +∞])` ; bornes supérieures incluses ; mensualité absente/invalide → `Non renseigné` |
| 22 | `monthly_fee_change_pct` | Évolution du tarif du même client depuis le mois précédent | `100 × (monthly_fee_t − monthly_fee_t−1) / monthly_fee_t−1` ; uniquement deux mois consécutifs et tarif précédent > 0 |

**Lecture des formules :** `1[condition]` vaut 1 si la condition est vraie, 0 sinon. Les croisements de catégories utilisent une concaténation avec `"__"`, pas une multiplication.

**Valeurs invalides et manquantes :** une ancienneté négative est remplacée par une valeur manquante. Avant le calcul des variables dérivées, les appels, retards, montants et consommations négatifs ainsi que les NPS hors de `[0,10]` sont traités comme manquants. Si une entrée nécessaire est manquante, le résultat dérivé reste manquant, y compris pour les indicateurs binaires ; exceptions : `nps_missing` vaut alors 1 et `monthly_fee_band` prend la catégorie `Non renseigné`. Une division par zéro ou un ratio non fini produit une valeur manquante. Un croisement catégoriel reste manquant si l'une de ses deux composantes manque. Ces règles concernent les variables dérivées et ne réécrivent pas les colonnes sources.


**Contrainte juridique :** ni `age` ni `age_band` ne sont utilisés dans la décision. Une tranche d’âge reste une transformation de l’âge. `retention_offer_declined` est exclu, faute de chronologie fiable de l’intervention commerciale.

**Important :** le modèle peut utiliser `monthly_fee` pour prédire le churn, mais la décision économique de campagne sera traitée séparément dans une étape ultérieure.

### 3.1 — Définir les transformations

Les transformations métier existantes sont déterministes. La nouvelle variable `monthly_fee_band` a des bornes apprises : les tertiles des mensualités valides des snapshots du **train janvier–juillet seulement**, sans utiliser la cible. Le montant numérique `monthly_fee` est conservé. Les mêmes bornes sont appliquées à validation et test ; chaque diagnostic par client ou fold temporel utilise des bornes ajustées sur son propre train.

Les groupes « Peu coûteux », « Intermédiaire » et « Coûteux » sont relatifs au train, pas des gammes commerciales officielles. Une valeur manquante, négative ou non finie devient « Non renseigné ». En cas de seuils confondus, le nombre de groupes est réduit ; aucun découpage artificiel des ex æquo. Les bornes extrêmes infinies couvrent les prix futurs hors plage du train.

`ChurnFeatureEngineer` orchestre les six petites classes. On ne crée ni n’ajuste `MonthlyFeeBands` dans les cellules d’utilisation : cet ajustement est géré à l’intérieur de la grande classe.

**Variation tarifaire :** `MonthlyFeeChanges` est gérée à l'intérieur de `ChurnFeatureEngineer`. Elle trie les observations par client et date, puis compare uniquement deux mois calendaires consécutifs. Exemple : 40 € → 48 € donne +20 %. Le premier mois sans historique, un trou de suivi, un tarif précédent nul, ou un montant absent/négatif/non fini donnent une valeur manquante. Un tarif actuel nul avec un précédent positif donne −100 %.

L'historique mémorisé par `fit` contient seulement identifiant, date et prix du train. `transform` peut aussi recevoir les snapshots historiques disponibles avant le scoring : aucun prix futur ni label ne sert au calcul d'une ligne passée. Aucun remplissage avec un mois plus ancien n'est effectué. L'ordre et l'index d'entrée restent conservés.


In [19]:
def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    value = numerator / denominator.where(denominator.gt(0))
    return value.replace([np.inf, -np.inf], np.nan)


class LifecycleFeatures:
    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        x = frame.copy()

        months = (x["snapshot_date"] - x["signup_date"]).dt.days / 30.4375
        months = months.where(months.ge(0))

        x["tenure_months"] = months
        x["tenure_band"] = pd.cut(
            months,
            bins=[0, 3, 6, 12, 24, 48, np.inf],
            right=False,
            labels=["0-3m", "3-6m", "6-12m", "12-24m", "24-48m", "48m+"],
        ).astype("string")
        x["is_new_customer"] = months.lt(3).where(months.notna()).astype("Int64")
        return x


class FrictionFeatures:
    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        x = frame.copy()
        calls = x["support_calls_3m"].where(x["support_calls_3m"].ge(0))
        late = x["late_payments"].where(x["late_payments"].ge(0))

        x["has_support_calls"] = calls.gt(0).where(calls.notna()).astype("Int64")
        x["high_support_calls"] = calls.ge(3).where(calls.notna()).astype("Int64")
        x["has_late_payments"] = late.gt(0).where(late.notna()).astype("Int64")
        x["support_payment_interaction"] = calls * late
        x["high_friction"] = (
            (calls.ge(3) & late.ge(1)).where(calls.notna() & late.notna()).astype("Int64")
        )
        return x


class SatisfactionFeatures:
    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        x = frame.copy()
        score = x["nps"].where(x["nps"].between(0, 10))
        calls = x["support_calls_3m"].where(x["support_calls_3m"].ge(0))

        x["nps_segment"] = pd.cut(
            score,
            bins=[-0.1, 6, 8, 10],
            labels=["detractor", "passive", "promoter"],
        ).astype("string")
        x["support_dissatisfaction"] = calls * (10 - score)
        x["nps_missing"] = score.isna().astype("int8")
        return x


class BillingFeatures:
    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        x = frame.copy()
        fee = x["monthly_fee"].where(x["monthly_fee"].ge(0))
        invoice = x["last_invoice_amount"].where(x["last_invoice_amount"].ge(0))
        usage = x["data_gb"].where(x["data_gb"].ge(0))
        late = x["late_payments"].where(x["late_payments"].ge(0))

        x["invoice_fee_gap"] = invoice - fee
        x["invoice_fee_ratio"] = safe_ratio(invoice, fee)
        x["fee_per_gb"] = safe_ratio(fee, usage)
        x["low_usage_flag"] = usage.lt(1).where(usage.notna()).astype("Int64")
        x["payment_invoice_interaction"] = late * (invoice - fee).clip(lower=0)
        x["invoice_below_fee"] = (
            invoice.lt(fee).where(invoice.notna() & fee.notna()).astype("Int64")
        )
        return x


class CommercialFeatures:
    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        x = frame.copy()
        pairs = {
            "contract_tenure": ("contract", "tenure_band"),
            "contract_channel": ("contract", "channel"),
            "contract_region": ("contract", "region"),
        }
        for name, (left, right) in pairs.items():
            a = x[left].astype("string")
            b = x[right].astype("string")
            x[name] = (a + "__" + b).where(a.notna() & b.notna())
        return x


class MonthlyFeeChanges:
    """Variation tarifaire d'un client entre deux mois consécutifs, sans valeur future."""

    _columns = ["customer_id", "snapshot_date", "monthly_fee"]

    def fit(self, train_frame: pd.DataFrame):
        # Historique de covariables uniquement : aucun label n'est mémorisé.
        self._history = train_frame[self._columns].copy()
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        result = frame.copy()
        current = frame[self._columns].copy()
        history = pd.concat([self._history, current], ignore_index=True)
        keys = ["customer_id", "snapshot_date"]
        if current.duplicated(keys).any():
            raise ValueError("Clés client × snapshot répétées pour la variation tarifaire.")
        history = history.drop_duplicates(keys, keep="last").sort_values(keys)
        if history[keys].isna().any().any():
            raise ValueError("Identifiant ou date manquante pour la variation tarifaire.")
        months = history.snapshot_date.dt.to_period("M")
        if history.assign(month=months).duplicated(["customer_id", "month"]).any():
            raise ValueError("Plusieurs snapshots du même client dans un mois.")
        fee = pd.Series(pd.to_numeric(history.monthly_fee, errors="coerce"), index=history.index)
        fee = fee.where(fee.ge(0) & np.isfinite(fee))
        previous_fee = fee.groupby(history.customer_id).shift(1)
        month_number = history.snapshot_date.dt.year * 12 + history.snapshot_date.dt.month
        previous_month = month_number.groupby(history.customer_id).shift(1)
        consecutive = month_number.sub(previous_month).eq(1)
        change = 100 * (fee - previous_fee) / previous_fee.where(previous_fee.gt(0))
        history["monthly_fee_change_pct"] = change.where(consecutive).replace(
            [np.inf, -np.inf], np.nan
        )
        # Revenir exactement à l'ordre et à l'index du DataFrame reçu.
        change_by_key = history.set_index(keys)["monthly_fee_change_pct"]
        result["monthly_fee_change_pct"] = change_by_key.reindex(
            pd.MultiIndex.from_frame(current[keys])
        ).to_numpy()
        return result


class MonthlyFeeBands:
    """Tranches tarifaires ajustées sur les mensualités valides du train seulement."""

    def fit(self, frame: pd.DataFrame):
        fee = pd.Series(pd.to_numeric(frame["monthly_fee"], errors="coerce"), index=frame.index)
        fee = fee.where(fee.ge(0) & np.isfinite(fee)).dropna()
        if fee.empty:
            raise ValueError("Aucune mensualité valide dans le train pour définir les tranches.")
        cuts = np.unique(fee.quantile([1 / 3, 2 / 3]).to_numpy(dtype=float))
        cuts = cuts[(cuts > fee.min()) & (cuts < fee.max())]
        self.edges_ = np.r_[-np.inf, cuts, np.inf]
        self.labels_ = {
            1: ["Intermédiaire"],
            2: ["Peu coûteux", "Coûteux"],
            3: ["Peu coûteux", "Intermédiaire", "Coûteux"],
        }[len(self.edges_) - 1]
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        if not hasattr(self, "edges_"):
            raise ValueError("Appeler fit sur le train avant transform.")
        result = frame.copy()
        fee = pd.Series(pd.to_numeric(result["monthly_fee"], errors="coerce"), index=result.index)
        fee = fee.where(fee.ge(0) & np.isfinite(fee))
        result["monthly_fee_band"] = (
            pd.Series(
                pd.cut(
                    fee,
                    bins=self.edges_,
                    labels=self.labels_,
                    right=True,
                ),
                index=result.index,
            )
            .astype("string")
            .fillna("Non renseigné")
        )
        return result


NEW_FEATURES = [
    "tenure_months",
    "tenure_band",
    "is_new_customer",
    "has_support_calls",
    "high_support_calls",
    "has_late_payments",
    "support_payment_interaction",
    "high_friction",
    "nps_segment",
    "support_dissatisfaction",
    "nps_missing",
    "invoice_fee_gap",
    "invoice_fee_ratio",
    "fee_per_gb",
    "low_usage_flag",
    "contract_tenure",
    "contract_channel",
    "contract_region",
    "payment_invoice_interaction",
    "invoice_below_fee",
    "monthly_fee_band",
    "monthly_fee_change_pct",
]


class ChurnFeatureEngineer:
    """Point d'entrée unique : ajuste et applique toutes les transformations métier."""

    def __init__(self):
        self._fee_bands = MonthlyFeeBands()
        self._fee_changes = MonthlyFeeChanges()
        self._transformers = [
            LifecycleFeatures(),
            FrictionFeatures(),
            SatisfactionFeatures(),
            BillingFeatures(),
            CommercialFeatures(),
            self._fee_bands,
            self._fee_changes,
        ]
        self._is_fitted = False

    def fit(self, train_frame: pd.DataFrame):
        """Ajuster les bornes et conserver les tarifs historiques du train, sans labels."""
        self._is_fitted = False
        self._fee_bands.fit(train_frame)
        self._fee_changes.fit(train_frame)
        self._is_fitted = True
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Appliquer toutes les petites classes avec les paramètres déjà appris."""
        if not self._is_fitted:
            raise ValueError("Appeler ChurnFeatureEngineer.fit sur le train avant transform.")
        result = frame.copy()
        for transformer in self._transformers:
            result = transformer.transform(result)
        expected_columns = set(frame.columns) | set(NEW_FEATURES)
        if set(result.columns) != expected_columns:
            raise AssertionError(f"Colonnes inattendues : {set(result.columns) ^ expected_columns}")
        return result

    def fit_transform(self, train_frame: pd.DataFrame) -> pd.DataFrame:
        """Ajuster puis transformer le train en un appel."""
        return self.fit(train_frame).transform(train_frame)

    def get_feature_metadata(self) -> dict:
        """Exposer les paramètres nécessaires à MLflow sans manipuler les sous-classes."""
        if not self._is_fitted:
            raise ValueError("Ajuster ChurnFeatureEngineer avant de lire ses métadonnées.")
        return {
            "monthly_fee_change_pct": {
                "formula": "100 * (monthly_fee_t - monthly_fee_t_minus_1) / monthly_fee_t_minus_1",
                "groupby": "customer_id",
                "previous": "consecutive_calendar_month",
                "missing": "NaN",
                "history": "past_covariates_only",
            },
            "monthly_fee_band": {
                "edges": [float(v) if np.isfinite(v) else None for v in self._fee_bands.edges_],
                "labels": list(self._fee_bands.labels_),
                "right_closed": True,
                "missing_label": "Non renseigné",
                "fit_source": "train_only",
            },
        }

### 3.2 — Appeler la classe principale pour toutes les transformations

Un seul objet, `feature_engineer`, s'occupe de l'ensemble des variables. `fit(feature_train)` ajuste les bornes sur le train, puis `transform(labeled)` applique toutes les transformations avec ces mêmes bornes. Pour transformer uniquement le train en un appel, la classe propose aussi `fit_transform(train_frame)`.

Chaque diagnostic utilise sa propre instance de cette même classe, ajustée sur son train. Les sous-classes restent un détail interne ; leurs métadonnées sont exposées par `get_feature_metadata()` pour MLflow.

In [20]:
# La grande classe ajuste ses sous-transformations sur le train uniquement.
feature_train = labeled.loc[labeled["snapshot_date"].dt.to_period("M").between(*TRAIN_MONTHS)]
feature_engineer = ChurnFeatureEngineer()
feature_engineer.fit(feature_train)
featured = feature_engineer.transform(labeled)

# La classe expose aussi les informations utiles à l'analyse et à MLflow.
fee_band_metadata = feature_engineer.get_feature_metadata()["monthly_fee_band"]
assert len(NEW_FEATURES) == 22
print("Bornes tarifaires apprises sur le train (€) :", fee_band_metadata["edges"])
print("Groupes :", fee_band_metadata["labels"])
print("Nouvelles variables créées :", len(NEW_FEATURES))
display(featured[NEW_FEATURES].head())

Bornes tarifaires apprises sur le train (€) : [None, 52.78, 68.2333333333333, None]
Groupes : ['Peu coûteux', 'Intermédiaire', 'Coûteux']
Nouvelles variables créées : 22


,tenure_months,tenure_band,is_new_customer,has_support_calls,high_support_calls,has_late_payments,support_payment_interaction,high_friction,nps_segment,support_dissatisfaction,...,invoice_fee_ratio,fee_per_gb,low_usage_flag,contract_tenure,contract_channel,contract_region,payment_invoice_interaction,invoice_below_fee,monthly_fee_band,monthly_fee_change_pct
0,46.882957,24-48m,0,1,0,1,1,0,passive,3.0,...,1.0,8.246835,0,biannuel__24-48m,biannuel__partenaire,biannuel__occ,0.0,0,Intermédiaire,0.773395
1,47.178645,24-48m,0,0,0,0,0,0,<NA>,NaN,...,1.0,3.395070,0,mensuel__24-48m,mensuel__telephone,mensuel__ara,0.0,0,Peu coûteux,-8.258801
2,3.778234,3-6m,0,1,0,0,0,0,promoter,0.0,...,1.0,8.463793,0,mensuel__3-6m,mensuel__partenaire,mensuel__idf,0.0,0,Peu coûteux,-3.194636
3,37.913758,24-48m,0,1,1,0,0,0,<NA>,NaN,...,1.0,2.283712,0,mensuel__24-48m,mensuel__web,mensuel__occ,0.0,0,Intermédiaire,-3.966231
4,45.404517,24-48m,0,1,0,0,0,0,<NA>,NaN,...,1.0,28.938889,0,annuel__24-48m,annuel__telephone,annuel__naq,0.0,0,Peu coûteux,-7.081698


### 3.3 — Choisir les entrées du modèle

Les catégories sont transmises telles quelles à CatBoost. Les conversions suivantes sont uniquement techniques : chaînes pour les catégories, nombres flottants pour les variables numériques, `NaN` conservés pour les numériques. Aucune transformation supervisée sur tout le dataset.

`invoice_fee_ratio` est conservée dans les données pour audit, mais exclue des prédicteurs afin de garder `invoice_fee_gap` comme représentation de l’écart de facture. `monthly_fee_change_pct` est une entrée numérique distincte, qui mesure le changement tarifaire dans le temps.

In [21]:
ORIGINAL_FEATURES = [
    "contract",
    "region",
    "dept",
    "channel",
    "monthly_fee",
    "data_gb",
    "support_calls_3m",
    "late_payments",
    "nps",
    "last_invoice_amount",
]

CATEGORICAL_FEATURES = [
    "contract",
    "region",
    "dept",
    "channel",
    "tenure_band",
    "nps_segment",
    "contract_tenure",
    "contract_channel",
    "contract_region",
    "monthly_fee_band",
]

EXCLUDED = [
    "customer_id",
    "snapshot_date",
    "signup_date",
    "age",
    "retention_offer_declined",
    "churn",
    "invoice_below_fee",
    "invoice_fee_ratio",
]

# Conserver toutes les variables créées dans les données, mais filtrer les entrées du modèle.
ORIGINAL_FEATURES = [f for f in ORIGINAL_FEATURES if f not in EXCLUDED]
SELECTED_NEW_FEATURES = [f for f in NEW_FEATURES if f not in EXCLUDED]
MODEL_FEATURES = ORIGINAL_FEATURES + SELECTED_NEW_FEATURES
CATEGORICAL_FEATURES = [f for f in CATEGORICAL_FEATURES if f in MODEL_FEATURES]

assert MODEL_FEATURES, "Aucune variable retenue pour le modèle."
assert not set(MODEL_FEATURES) & set(EXCLUDED)
assert len(set(MODEL_FEATURES)) == len(MODEL_FEATURES)

print("Variables originales :", len(ORIGINAL_FEATURES))
print("Nouvelles variables retenues :", len(SELECTED_NEW_FEATURES))
print("Total :", len(MODEL_FEATURES))

display(
    pd.DataFrame(
        {
            "variable": MODEL_FEATURES,
            "source": ["Originale"] * len(ORIGINAL_FEATURES)
            + ["Créée"] * len(SELECTED_NEW_FEATURES),
        }
    )
)

Variables originales : 10
Nouvelles variables retenues : 20
Total : 30


,variable,source
0,contract,Originale
1,region,Originale
2,dept,Originale
3,channel,Originale
4,monthly_fee,Originale
5,data_gb,Originale
6,support_calls_3m,Originale
7,late_payments,Originale
8,nps,Originale
9,last_invoice_amount,Originale


### 3.4 — Auditer les variables créées

Le contrôle de qualité n’élimine pas automatiquement les variables. Une variable très manquante ou constante devra être discutée ; l’absence de valeur n’est pas une raison suffisante pour inventer une imputation.

In [22]:
audit = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "missing_pct": [featured[c].isna().mean() * 100 for c in MODEL_FEATURES],
        "distinct_values": [featured[c].nunique(dropna=True) for c in MODEL_FEATURES],
    }
)

display(audit.sort_values("missing_pct", ascending=False))

fig = px.bar(
    audit.sort_values("missing_pct", ascending=True),
    x="missing_pct",
    y="feature",
    orientation="h",
    title="Valeurs manquantes par variable (%)",
    color="missing_pct",
    color_continuous_scale="Blues",
    height=750,
)
fig.update_layout(template="plotly_white", coloraxis_showscale=False)
fig.show(config={"responsive": True})

,feature,missing_pct,distinct_values
8,nps,67.829869,11
18,nps_segment,67.829869,3
19,support_dissatisfaction,67.829869,35
29,monthly_fee_change_pct,19.392167,30981
22,fee_per_gb,5.260297,34475
27,payment_invoice_interaction,5.200756,1
21,invoice_fee_gap,5.200756,1061
4,monthly_fee,4.908230,8273
24,contract_tenure,0.447850,18
10,tenure_months,0.447850,2175


## 4 — Split temporel

- **Train** : janvier–juillet 2024 ; **Validation** : août–septembre 2024 ; **Test** : octobre–novembre 2024.
- Ce protocole évalue une **généralisation temporelle hors échantillon**, mais **ne simule pas un modèle unique déployé au 31 octobre** : les labels de septembre n'étaient pas disponibles au début d'octobre. La validation septembre peut être utilisée pour sélectionner le modèle avant le test de novembre, **pas pour une prédiction d'octobre au 31 octobre** sans vérification des dates exactes de disponibilité.
- Le test n'est utilisé ni pour sélectionner les features ni pour ajuster le seuil. Les résultats d'octobre et novembre sont une évaluation rétrospective ; une simulation walk-forward indépendante est proposée plus loin.
- La répétition des `customer_id` entre mois est **attendue** ; le `customer_id` n'est jamais une feature.

In [23]:
months = featured["snapshot_date"].dt.to_period("M")

train_df = featured.loc[months.between(*TRAIN_MONTHS)].copy()
val_df = featured.loc[months.between(*VALIDATION_MONTHS)].copy()
test_df = featured.loc[months.between(*TEST_MONTHS)].copy()

splits = {"Train": train_df, "Validation": val_df, "Test": test_df}

summary = pd.DataFrame(
    [
        {
            "split": name,
            "snapshots": len(part),
            "clients": part["customer_id"].nunique(),
            "positives": int(part["churn"].sum()),
            "churn_rate": part["churn"].mean(),
        }
        for name, part in splits.items()
    ]
)

display(summary)

for name, part in splits.items():
    if len(part) == 0 or part["churn"].nunique() != 2:
        raise ValueError(f"{name} vide ou avec une seule classe : revoir les périodes.")

assert train_df["snapshot_date"].max() < val_df["snapshot_date"].min()
assert val_df["snapshot_date"].max() < test_df["snapshot_date"].min()

fig = px.bar(
    summary,
    x="split",
    y="snapshots",
    color="split",
    text="snapshots",
    title="Répartition chronologique des snapshots",
)
fig.update_layout(template="plotly_white", showlegend=False, height=380)
fig.show(config={"responsive": True})

,split,snapshots,clients,positives,churn_rate
0,Train,23322,3809,703,0.030143
1,Validation,6309,3239,209,0.033127
2,Test,6091,3088,181,0.029716


### 4.1 — Vérifier les chevauchements de clients

Une intersection non vide entre les ensembles de clients **n'est pas une fuite à elle seule**. Le risque dépend de l'ordre des snapshots, des dates de connaissance des labels et du calcul des features. Ici, aucune information future n'est utilisée par les transformations déterministes ; il reste à confirmer la disponibilité opérationnelle des variables de facture et d'offre.

In [24]:
sets = {name: set(part["customer_id"].dropna()) for name, part in splits.items()}
intersections = pd.DataFrame(
    [
        {
            "partitions": "Train / Validation",
            "clients_communs": len(sets["Train"] & sets["Validation"]),
        },
        {"partitions": "Train / Test", "clients_communs": len(sets["Train"] & sets["Test"])},
        {
            "partitions": "Validation / Test",
            "clients_communs": len(sets["Validation"] & sets["Test"]),
        },
    ]
)
display(intersections)
dates_audit = pd.DataFrame(
    [
        {
            "split": name,
            "min_snapshot": part.snapshot_date.min(),
            "max_snapshot": part.snapshot_date.max(),
            "clients": part.customer_id.nunique(),
            "observations": len(part),
        }
        for name, part in splits.items()
    ]
)
display(dates_audit)
fig = px.bar(
    intersections,
    x="partitions",
    y="clients_communs",
    text="clients_communs",
    title="Clients communs entre partitions (attendus en panel)",
    color_discrete_sequence=["#7C3AED"],
)
fig.update_layout(template="plotly_white", height=370)
fig.show(config={"responsive": True})

,partitions,clients_communs
0,Train / Validation,3106
1,Train / Test,2918
2,Validation / Test,3030


,split,min_snapshot,max_snapshot,clients,observations
0,Train,2024-01-31,2024-07-31,3809,23322
1,Validation,2024-08-31,2024-09-30,3239,6309
2,Test,2024-10-31,2024-11-30,3088,6091


### 4.2 — Lire les tranches tarifaires

Les bornes sont déjà figées sur le train ; les tableaux ci-dessous portent sur **train et validation**, sans consulter les taux du test pour choisir les seuils. Ils indiquent effectifs, clients distincts, churns, taux de churn, part des snapshots et lift par rapport au taux global du split. Un lift supérieur à 1 indique une surreprésentation descriptive, pas un effet causal du prix. Les clients répétés pèsent plusieurs fois ; un faible effectif rend le taux fragile. Les analyses de classement se jugent ensuite avec AP, rappel et précision, pas avec ce regroupement seul.

In [25]:
fee_band_tables = []
fee_band_order = fee_band_metadata["labels"] + ["Non renseigné"]
for split_name, part in [("Train", train_df), ("Validation", val_df)]:
    table = (
        part.groupby("monthly_fee_band", observed=True)
        .agg(
            snapshots=("churn", "size"),
            clients=("customer_id", "nunique"),
            churns=("churn", "sum"),
            churn_rate=("churn", "mean"),
            minimum_fee=("monthly_fee", "min"),
            maximum_fee=("monthly_fee", "max"),
        )
        .reindex(fee_band_order)
    )
    table = table.loc[table.snapshots.notna()].copy()
    table["share_snapshots"] = table.snapshots / len(part)
    table["lift"] = table.churn_rate / part.churn.mean()
    table["split"] = split_name
    fee_band_tables.append(table.reset_index())
fee_band_summary = pd.concat(fee_band_tables, ignore_index=True)
display(fee_band_summary)
fig = px.bar(
    fee_band_summary,
    x="monthly_fee_band",
    y="churn_rate",
    color="split",
    barmode="group",
    category_orders={"monthly_fee_band": fee_band_order},
    custom_data=["snapshots", "clients", "churns", "lift"],
    title="Churn par tranche tarifaire — bornes ajustées sur le train",
)
fig.update_traces(
    hovertemplate="%{x}<br>Churn : %{y:.2%}<br>Snapshots : %{customdata[0]}<br>Clients : %{customdata[1]}<br>Churns : %{customdata[2]}<br>Lift : %{customdata[3]:.2f}<extra></extra>"
)
fig.update_yaxes(tickformat=".1%", title="Taux de churn par snapshot")
fig.update_layout(template="plotly_white", xaxis_title="Tranche de monthly_fee")
fig.show(config={"responsive": True})

,monthly_fee_band,snapshots,clients,churns,churn_rate,minimum_fee,maximum_fee,share_snapshots,lift,split
0,Peu coûteux,7372,1432,181,0.024552,10.34,52.78,0.316096,0.814524,Train
1,Intermédiaire,7368,1639,209,0.028366,52.79,68.23,0.315925,0.941038,Train
2,Coûteux,7370,1497,271,0.036771,68.24,128.25,0.316011,1.219866,Train
3,Non renseigné,1212,1057,42,0.034653,-90.24,-18.94,0.051968,1.149627,Train
4,Peu coûteux,1852,1067,51,0.027538,11.30,52.78,0.293549,0.831273,Validation
5,Intermédiaire,1971,1188,67,0.033993,52.79,68.22,0.312411,1.026130,Validation
6,Coûteux,2154,1220,80,0.037140,68.25,130.39,0.341417,1.121137,Validation
7,Non renseigné,332,324,11,0.033133,-109.28,-54.10,0.052623,1.000159,Validation


## 5 — Sauvegarder les données et les rapports

On enregistre les trois partitions enrichies sans index et leurs SHA-256. Les rapports et métadonnées sont séparés des CSV dans `data/interim/features/`. Relancer le notebook remplace ces préparations ; les futurs runs MLflow du 02 en archiveront leurs propres copies.

In [26]:
def save_prepared_frame(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return {
        "path": path.relative_to(settings.PROJECT_ROOT).as_posix(),
        "rows": len(frame),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }


PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
prepared_datasets = {
    name: save_prepared_frame(frame, settings.PROCESSED_DIR / f"{name}.csv")
    for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]
}
preparation_reports = {
    "feature_quality": audit,
    "split_summary": summary,
    "monthly_fee_bands": fee_band_summary,
    "client_intersections": intersections,
    "panel_audit": panel_audit,
    "selected_features": pd.DataFrame(
        {
            "feature": MODEL_FEATURES,
            "categorical": [feature in CATEGORICAL_FEATURES for feature in MODEL_FEATURES],
        }
    ),
}
report_files = {}
for name, table in preparation_reports.items():
    report_files[name] = save_prepared_frame(table, PREPARATION_DIR / "reports" / f"{name}.csv")
print("Partitions principales sauvegardées dans :", settings.PROCESSED_DIR)

Partitions principales sauvegardées dans : /Users/surelmanda/Downloads/sujet-entretien-technique/Telecom-Retention-Optimize/data/processed


## 6 — Préparer les features des partitions de diagnostic

Pour conserver les protocoles existants sans refaire de feature engineering dans le 02, on prépare ici leurs entrées. Chaque partition par client et chaque fold temporel ajuste **sa propre instance** de `ChurnFeatureEngineer` sur son train, puis transforme son jeu de contrôle. Aucun entraînement prédictif n'est lancé. Ces CSV restent dans `data/interim/features/diagnostics/` et ne remplacent pas les trois partitions principales.

Pour la variation tarifaire, le contexte historique complet est transformé chronologiquement, puis les lignes du train et du contrôle sont sélectionnées. Ainsi un mois purgé pour ses labels peut fournir son tarif déjà connu ; les labels et les observations futures ne participent pas à cette transformation déterministe.

In [27]:
def prepare_diagnostic_inputs(train_source, validation_source, name):
    engineer = ChurnFeatureEngineer()
    engineer.fit(train_source)
    # Le prix du mois précédent peut être connu même si son label est purgé.
    # Le calcul temporel utilise seulement les covariables antérieures du même client.
    transformed_history = engineer.transform(labeled)
    transformed_train = transformed_history.loc[train_source.index].copy()
    transformed_validation = transformed_history.loc[validation_source.index].copy()
    directory = PREPARATION_DIR / "diagnostics" / name
    return {
        "datasets": {
            "train": save_prepared_frame(transformed_train, directory / "train.csv"),
            "validation": save_prepared_frame(transformed_validation, directory / "validation.csv"),
        },
        "feature_metadata": engineer.get_feature_metadata(),
    }


pretest = labeled.loc[
    labeled.snapshot_date.dt.to_period("M") < pd.Period(TEST_MONTHS[0], freq="M")
].copy()
group_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_positions, holdout_positions = next(
    group_split.split(pretest, pretest.churn, groups=pretest.customer_id)
)
group_source_train, group_source_holdout = (
    pretest.iloc[train_positions],
    pretest.iloc[holdout_positions],
)
assert set(group_source_train.customer_id).isdisjoint(set(group_source_holdout.customer_id))
prepared_diagnostics = {
    "clients": prepare_diagnostic_inputs(group_source_train, group_source_holdout, "clients"),
    "walk_forward": [],
}
periods = labeled.snapshot_date.dt.to_period("M")
for test_month in pd.period_range(*WALK_FORWARD_MONTHS, freq="M"):
    last_train_month = test_month - 2
    fold_source_train = labeled.loc[periods <= last_train_month]
    fold_source_validation = labeled.loc[periods == test_month]
    if (
        fold_source_train.empty
        or fold_source_validation.empty
        or fold_source_train.churn.nunique() < 2
        or fold_source_validation.churn.nunique() < 2
    ):
        print("Partition ignorée :", test_month)
        continue
    entry = prepare_diagnostic_inputs(
        fold_source_train, fold_source_validation, f"walk_{test_month}"
    )
    entry.update(month=str(test_month), last_train_month=str(last_train_month))
    prepared_diagnostics["walk_forward"].append(entry)
print("Diagnostic clients préparé ; folds temporels :", len(prepared_diagnostics["walk_forward"]))

Diagnostic clients préparé ; folds temporels : 4


## 7 — Écrire le contrat de préparation pour le notebook 02

Le contrat décrit les fichiers, leurs empreintes, les colonnes et les bornes de transformation. Il conserve aussi une copie du code du notebook de features sans ses sorties. Le 02 vérifie ces fichiers avant de les utiliser ; il ne recrée aucune feature et ne recopie aucune classe de transformation.

**Suite :** ouvrir `02_churn_modelisation.ipynb` dans un noyau neuf.

In [28]:
feature_snapshot = json.loads(FEATURE_NOTEBOOK_PATH.read_text())
for cell in feature_snapshot["cells"]:
    if cell["cell_type"] == "code":
        cell["outputs"] = []
        cell["execution_count"] = None
source_path = PREPARATION_DIR / "source" / "notebook_01.ipynb"
source_path.parent.mkdir(parents=True, exist_ok=True)
source_path.write_text(json.dumps(feature_snapshot, ensure_ascii=False, indent=1) + "\n")
feature_contract = {
    "preparation_schema_version": 1,
    "seed": SEED,
    "model_features": MODEL_FEATURES,
    "original_features": ORIGINAL_FEATURES,
    "new_features": NEW_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "excluded": EXCLUDED,
    "feature_metadata": feature_engineer.get_feature_metadata(),
    "periods": {
        "train": TRAIN_MONTHS,
        "validation": VALIDATION_MONTHS,
        "test": TEST_MONTHS,
        "walk_forward": WALK_FORWARD_MONTHS,
    },
    "datasets": prepared_datasets,
    "diagnostics": prepared_diagnostics,
    "reports": report_files,
    "quality": {
        "raw_rows": raw_count,
        "exact_duplicates_removed": exact_removed,
        "ambiguous_keys_excluded": len(ambiguous_keys),
        "retained_labeled_rows": len(labeled),
        "source_csv": str(settings.DATA_PATH),
        "source_sha256": hashlib.sha256(settings.DATA_PATH.read_bytes()).hexdigest(),
    },
    "source": {
        "path": source_path.relative_to(settings.PROJECT_ROOT).as_posix(),
        "sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(),
    },
}
feature_contract["preparation_id"] = hashlib.sha256(
    json.dumps(feature_contract, sort_keys=True, ensure_ascii=False).encode()
).hexdigest()
contract_path = PREPARATION_DIR / "feature_contract.json"
contract_path.write_text(json.dumps(feature_contract, ensure_ascii=False, indent=2) + "\n")
print("Préparation terminée :", feature_contract["preparation_id"])
print("Contrat :", contract_path)
print("Aucun modèle entraîné. Suite : notebook 02.")

Préparation terminée : d20df3e04b951dabd9f2f0cba960355eb0b2e951bc0a871f49b20973a64ee40d
Contrat : /Users/surelmanda/Downloads/sujet-entretien-technique/Telecom-Retention-Optimize/data/interim/features/feature_contract.json
Aucun modèle entraîné. Suite : notebook 02.
